# Assignment 02: Fetching Occurrence Records With pygbif

## BIO597 Spatial Analysis of Biodiversity

This assignment gives you more practice with the workflow from Lab 02.

You will use `pygbif` to search GBIF for several snake species, download a small number of occurrence records, convert those records to `GeoDataFrame` objects, and make simple maps and summaries.

Some cells are partly filled in. You should fill in the missing pieces and run the notebook from top to bottom.

Remember, learning to code is often about learning how to strategically copy, paste, and modify. Look back at Lab 02 when you need a model.


## Species for this assignment

Use these species:

* *Storeria dekayi*
* *Agkistrodon contortrix*
* *Pantherophis guttatus*
* one additional snake species of your choice. Choosing: *Masticophis flagellum*

For each species, we will ask GBIF for georeferenced records so the results can be mapped.


## 1. Import packages

Import the same packages used in Lab 02.

`species` is used for taxonomic name searches. `occ` is used for occurrence record searches.


In [43]:
from pygbif import species
from pygbif import occurrences as occ

import pandas as pd
import geopandas as gpd

## 2. Search for possible name matches

Use `species.name_suggest()` to search for *Storeria dekayi*.

Fill in the species name. Then inspect the first result.


In [44]:
dekayi_suggestions = species.name_suggest(q="Storeria dekay")
# dekayi_suggestions will have a list of dictionaries
# select the first element of this list here and save it as a new variable called `dekayi_match`
dekayi_match = dekayi_suggestions[0]
dekayi_match

{'key': 9056579,
 'nameKey': 10782717,
 'kingdom': 'Animalia',
 'phylum': 'Chordata',
 'family': 'Colubridae',
 'genus': 'Storeria',
 'species': 'Storeria dekayi',
 'kingdomKey': 1,
 'phylumKey': 44,
 'classKey': 11592253,
 'familyKey': 6172,
 'genusKey': 9213370,
 'speciesKey': 9056579,
 'parent': 'Storeria',
 'parentKey': 9213370,
 'nubKey': 9056579,
 'scientificName': 'Storeria dekayi (Holbrook, 1839)',
 'canonicalName': 'Storeria dekayi',
 'rank': 'SPECIES',
 'status': 'ACCEPTED',
 'synonym': False,
 'higherClassificationMap': {'1': 'Animalia',
  '44': 'Chordata',
  '11592253': 'Squamata',
  '6172': 'Colubridae',
  '9213370': 'Storeria'},
 'class': 'Squamata'}

## 3. Save the taxon key

Pull the `speciesKey` out of the match result and save it as `dekayi_key`.


In [45]:
dekayi_key = dekayi_match["speciesKey"]
dekayi_key

9056579

## 4. Count records before downloading

Use `occ.count()` to count georeferenced GBIF records for *Storeria dekayi*.

This count tells you how many records GBIF has that match your search, not how many you will download in this assignment.


In [46]:
dekayi_count = occ.count(
    taxonKey=dekayi_key,
    isGeoreferenced=True,
)

dekayi_count

49473

## 5. Determine how many _total_ records there are for S. dekayi

The `isGeoreferenced` parameter determines whether occurrences with latlongs are returned.
Make a copy of the call to `occ.count()` as above, but change the `True` to `False`, which
will return only occurrences **without** latlongs. Capture the results in a new variable 
called `dekayi_nolatlong_count` and then add this to `dekayi_count` from the previous cell
to get the total number of records.

In [47]:
# How many total S. Dekayi records are there?
dekayi_nolatlong_count = occ.count(
    taxonKey=dekayi_key,
    isGeoreferenced=False,
)

print(f"There are {dekayi_count} occurrence records with lat/long information.")
print(f"There are {dekayi_nolatlong_count} occurrence records without lat/long information.")

dekayi_total_count = dekayi_count + dekayi_nolatlong_count

print(f"There are {dekayi_total_count} total occurrence records for \x1B[3mStoreria dekayi\x1B[0m.")


There are 49473 occurrence records with lat/long information.
There are 7073 occurrence records without lat/long information.
There are 56546 total occurrence records for Storeria dekayi.


## 6. Fetch up to 100 records

Use `occ.search()` to fetch occurrence records for *Storeria dekayi*.

Keep `limit=100`. Do not request more than 100 records for a species in this assignment.


In [48]:
dekayi_records = occ.search(
    speciesKey=9056579,         ## We got this speciesKey from step 3.
    hasCoordinate=True,
    hasGeospatialIssue=False,
    limit=100,
)

dekayi_records.keys()

dict_keys(['offset', 'limit', 'endOfRecords', 'count', 'results', 'facets'])

## 7. Turn the records into a table

The occurrence records are stored in the "results" key of the `dekayi_records` dictionary. Convert that list of records to a pandas DataFrame.


In [49]:
dekayi_df = pd.DataFrame(dekayi_records["results"])
dekayi_df.head()

,key,datasetKey,publishingOrgKey,datasetCategory,installationKey,hostingOrganizationKey,publishingCountry,protocol,lastCrawled,lastParsed,...,eventTime,identificationID,occurrenceRemarks,informationWithheld,projectId,dynamicProperties,vitality,lifeStage,identificationRemarks,sex
0,5938064620,50c9509d-22c7-4a22-a47d-8c48425ef4a7,28eb1a3f-1c15-4a95-931a-4af90ecb574d,[CitizenScience],997448a8-f762-11e1-a439-00145eb45e9a,28eb1a3f-1c15-4a95-931a-4af90ecb574d,US,DWC_ARCHIVE,2026-09-10T19:15:51.157+00:00,2026-09-11T05:22:11.912+00:00,...,16:48:33-06:00,745332273,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,5938201753,50c9509d-22c7-4a22-a47d-8c48425ef4a7,28eb1a3f-1c15-4a95-931a-4af90ecb574d,[CitizenScience],997448a8-f762-11e1-a439-00145eb45e9a,28eb1a3f-1c15-4a95-931a-4af90ecb574d,US,DWC_ARCHIVE,2026-09-10T19:15:51.157+00:00,2026-09-11T05:48:03.041+00:00,...,14:41:00-06:00,746093157,Body about 0.5 cm wide.,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,5938457152,50c9509d-22c7-4a22-a47d-8c48425ef4a7,28eb1a3f-1c15-4a95-931a-4af90ecb574d,[CitizenScience],997448a8-f762-11e1-a439-00145eb45e9a,28eb1a3f-1c15-4a95-931a-4af90ecb574d,US,DWC_ARCHIVE,2026-09-10T19:15:51.157+00:00,2026-09-11T05:36:08.412+00:00,...,04:52:51-06:00,745694353,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,5938490464,50c9509d-22c7-4a22-a47d-8c48425ef4a7,28eb1a3f-1c15-4a95-931a-4af90ecb574d,[CitizenScience],997448a8-f762-11e1-a439-00145eb45e9a,28eb1a3f-1c15-4a95-931a-4af90ecb574d,US,DWC_ARCHIVE,2026-09-10T19:15:51.157+00:00,2026-09-11T05:51:42.840+00:00,...,14:00:00-05:00,746922821,First snake of the year !!!,Coordinate uncertainty increased to 28734m at ...,NaN,NaN,NaN,NaN,NaN,NaN
4,5938606080,50c9509d-22c7-4a22-a47d-8c48425ef4a7,28eb1a3f-1c15-4a95-931a-4af90ecb574d,[CitizenScience],997448a8-f762-11e1-a439-00145eb45e9a,28eb1a3f-1c15-4a95-931a-4af90ecb574d,US,DWC_ARCHIVE,2026-09-10T19:15:51.157+00:00,2026-09-11T05:22:15.726+00:00,...,11:27:14-06:00,745678895,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## 8. Keep a small set of useful columns

Keep the columns needed for a map and a few simple summaries.

Fill in the latitude column name.


In [50]:
dekayi_small = dekayi_df[
    ["key", "scientificName", "decimalLongitude", "decimalLatitude", "year", "basisOfRecord", "countryCode"]
]

dekayi_small.head()

,key,scientificName,decimalLongitude,decimalLatitude,year,basisOfRecord,countryCode
0,5938064620,"Storeria dekayi (Holbrook, 1839)",-97.122430,33.241408,2026,HUMAN_OBSERVATION,US
1,5938201753,"Storeria dekayi (Holbrook, 1839)",-92.908814,34.619900,2026,HUMAN_OBSERVATION,US
2,5938457152,"Storeria dekayi (Holbrook, 1839)",-86.721496,33.458035,2026,HUMAN_OBSERVATION,US
3,5938490464,"Storeria dekayi (Holbrook, 1839)",-80.744301,35.172939,2026,HUMAN_OBSERVATION,US
4,5938606080,"Storeria dekayi (Holbrook, 1839)",-97.508688,35.238997,2026,HUMAN_OBSERVATION,US


## 9. Convert the table to a GeoDataFrame

Use the longitude and latitude columns to create point geometry. This is the same idea as Lab 01 and Lab 02.


In [51]:
dekayi_gdf = gpd.GeoDataFrame(
    dekayi_small,
    geometry=gpd.points_from_xy(dekayi_small["decimalLongitude"], dekayi_small["decimalLatitude"]),
    crs="EPSG:4326",
)

dekayi_gdf["Species"] = "Storeria dekayi"
dekayi_gdf.head()

,key,scientificName,decimalLongitude,decimalLatitude,year,basisOfRecord,countryCode,geometry,Species
0,5938064620,"Storeria dekayi (Holbrook, 1839)",-97.122430,33.241408,2026,HUMAN_OBSERVATION,US,POINT (-97.12243 33.24141),Storeria dekayi
1,5938201753,"Storeria dekayi (Holbrook, 1839)",-92.908814,34.619900,2026,HUMAN_OBSERVATION,US,POINT (-92.90881 34.6199),Storeria dekayi
2,5938457152,"Storeria dekayi (Holbrook, 1839)",-86.721496,33.458035,2026,HUMAN_OBSERVATION,US,POINT (-86.7215 33.45804),Storeria dekayi
3,5938490464,"Storeria dekayi (Holbrook, 1839)",-80.744301,35.172939,2026,HUMAN_OBSERVATION,US,POINT (-80.7443 35.17294),Storeria dekayi
4,5938606080,"Storeria dekayi (Holbrook, 1839)",-97.508688,35.238997,2026,HUMAN_OBSERVATION,US,POINT (-97.50869 35.239),Storeria dekayi


## 10. Map *Storeria dekayi*

Make an interactive map of the *Storeria dekayi* records.


In [52]:
dekayi_gdf.explore()

## 11. Repeat the workflow for *Agkistrodon contortrix*

Now repeat the same steps for eastern copperhead, *Agkistrodon contortrix*.

This cell should get the name suggestions and save the speciesKey.


In [53]:
contortrix_suggestions = species.name_suggest(q="Agkistrodon contortrix")

# Select the first element from the `contortrix_suggestions` list
contortrix_match = contortrix_suggestions[0]

# Get the `speciesKey`
contortrix_key = contortrix_match["speciesKey"]

contortrix_key

9215881

## 12. Counts for *Agkistrodon contortrix*

In [54]:
contortrix_count = occ.count(
    taxonKey=contortrix_key,
    isGeoreferenced=True,
)

contortrix_count

# How many total S. Dekayi records are there?
contortrix_nolatlong_count = occ.count(
    taxonKey=contortrix_key,
    isGeoreferenced=False,
)

print(f"There are {contortrix_count} occurrence records with lat/long information.")
print(f"There are {contortrix_nolatlong_count} occurrence records without lat/long information.")

contortrix_total_count = contortrix_count + contortrix_nolatlong_count

print(f"There are {contortrix_total_count} total occurrence records available for \x1B[3mAgkistrodon contortrix\x1B[0m.")


There are 23303 occurrence records with lat/long information.
There are 5811 occurrence records without lat/long information.
There are 29114 total occurrence records available for Agkistrodon contortrix.


## 13. Fetch and map *Agkistrodon contortrix*

Write code to fetch up to 100 records with coordinates, convert them to a table, convert that table to a GeoDataFrame, add a `Species` column, and map the result.

Use the same variable names shown in the comments. In `occ.search()`, use `hasCoordinate=True` and `hasGeospatialIssue=False`.


In [55]:
# Create contortrix_records with occ.search().
contortrix_records = occ.search(
    speciesKey = contortrix_key,
    hasCoordinate = True,
    hasGeospatialIssue = False,
    limit = 100,
)

# Create contortrix_df from contortrix_records["results"].
contortrix_df = pd.DataFrame(contortrix_records["results"])

# Create contortrix_small with the columns you want to keep.
contortrix_small = contortrix_df[
    ["key", "scientificName", "decimalLongitude", "decimalLatitude", "year", "basisOfRecord", "countryCode"]
]

# Create contortrix_gdf with gpd.GeoDataFrame().
contortrix_gdf = gpd.GeoDataFrame(
    contortrix_small,
    geometry = gpd.points_from_xy(contortrix_small["decimalLongitude"], contortrix_small["decimalLatitude"]),
    crs = "EPSG:4326",
)

# Add a Species column with the name Agkistrodon contortrix.
contortrix_gdf["Species"] = "Agkistrodon contortrix"

# Map contortrix_gdf with .explore().
contortrix_gdf.explore()

## 14. Repeat the workflow for *Pantherophis guttatus*

This time you will do a little more on your own.

First, use `species.name_suggest()` to get the `speciesKey` for *Pantherophis guttatus*.


In [56]:
guttatus_suggestions = species.name_suggest(q="Pantherophis guttatus")

# Select the first element from the `guttatus_suggestions` list
guttatus_match = guttatus_suggestions[0]

# Get the `speciesKey`
guttatus_key = guttatus_match["speciesKey"]

guttatus_key

2455615

## 15. Counts for *Pantherophis guttatus*

In [57]:
guttatus_count = occ.count(
    taxonKey=guttatus_key,
    isGeoreferenced=True,
)

guttatus_count

# How many total P. guttatus records are there?
guttatus_nolatlong_count = occ.count(
    taxonKey=guttatus_key,
    isGeoreferenced=False,
)

print(f"There are {guttatus_count} occurrence records with lat/long information.")
print(f"There are {guttatus_nolatlong_count} occurrence records without lat/long information.")

guttatus_total_count = guttatus_count + guttatus_nolatlong_count

print(f"There are {guttatus_total_count} total occurrence records available for \x1B[3mPantherophis guttatus\x1B[0m.")


There are 10142 occurrence records with lat/long information.
There are 3065 occurrence records without lat/long information.
There are 13207 total occurrence records available for Pantherophis guttatus.


## 16. Fetch and map *Pantherophis guttatus*

Fetch up to 100 georeferenced records and convert them to a GeoDataFrame.

Keep the cell simple. It is fine to copy and modify code from earlier cells.


In [58]:
# Create guttatus_records with occ.search().
guttatus_records = occ.search(
    speciesKey = guttatus_key,
    hasCoordinate = True,
    hasGeospatialIssue = False,
    limit = 100,
)

# Create guttatus_df from guttatus_records["results"].
guttatus_df = pd.DataFrame(guttatus_records["results"])

# Create guttatus_small with the columns you want to keep.
guttatus_small = guttatus_df[
    ["key", "scientificName", "decimalLongitude", "decimalLatitude", "year", "basisOfRecord", "countryCode"]
]

# Create guttatus_gdf with gpd.GeoDataFrame().
guttatus_gdf = gpd.GeoDataFrame(
    guttatus_small,
    geometry = gpd.points_from_xy(guttatus_small["decimalLongitude"], guttatus_small["decimalLatitude"]),
    crs = "EPSG:4326",
)

# Add a Species column with the name Pantherophis guttatus.
guttatus_gdf["Species"] = "Pantherophis guttatus"

# Map guttatus_gdf with .explore().
guttatus_gdf.explore()

## 17. Combine the three GeoDataFrames

Use `pd.concat()` to combine your three species GeoDataFrames.

Then inspect the first few rows.


In [59]:
gbif_snakes = pd.concat([dekayi_gdf, contortrix_gdf, guttatus_gdf])

gbif_snakes.head()

,key,scientificName,decimalLongitude,decimalLatitude,year,basisOfRecord,countryCode,geometry,Species
0,5938064620,"Storeria dekayi (Holbrook, 1839)",-97.122430,33.241408,2026,HUMAN_OBSERVATION,US,POINT (-97.12243 33.24141),Storeria dekayi
1,5938201753,"Storeria dekayi (Holbrook, 1839)",-92.908814,34.619900,2026,HUMAN_OBSERVATION,US,POINT (-92.90881 34.6199),Storeria dekayi
2,5938457152,"Storeria dekayi (Holbrook, 1839)",-86.721496,33.458035,2026,HUMAN_OBSERVATION,US,POINT (-86.7215 33.45804),Storeria dekayi
3,5938490464,"Storeria dekayi (Holbrook, 1839)",-80.744301,35.172939,2026,HUMAN_OBSERVATION,US,POINT (-80.7443 35.17294),Storeria dekayi
4,5938606080,"Storeria dekayi (Holbrook, 1839)",-97.508688,35.238997,2026,HUMAN_OBSERVATION,US,POINT (-97.50869 35.239),Storeria dekayi


## 18. Map all three species together

Use `.explore()` and color by `Species` so you can compare the three species on one map.


In [60]:
gbif_snakes.explore(column="Species", cmap="rainbow")

## 19. Count records by species

Use `groupby()` to count how many records you downloaded for each species.


In [61]:
gbif_snakes.groupby("Species").size()

Species
Agkistrodon contortrix    100
Pantherophis guttatus     100
Storeria dekayi           100
dtype: int64

## 20. Count records by basis of record

The `countryCode` field describes the general type of occurrence record.

Use `groupby()` to count records by `countryCode`.


In [62]:
gbif_snakes.groupby("countryCode").size()

countryCode
MX      2
US    298
dtype: int64

## 21. Choose one additional species

Choose one additional snake species and repeat the workflow.

Your species does not have to be in the local `EasternSnakes` CSV files. It only needs to be a snake species that GBIF can find.

Your code should:

* use `species.name_suggest()`
* save the taxon key
* count georeferenced records
* fetch no more than 100 records with `occ.search()`
* convert the records to a `GeoDataFrame`
* map the records


In [63]:
## Selected species: Thamnophis saurita (Eastern Ribbon Snake)
saurita_suggestions = species.name_suggest(q="Thamnophis saurita")

## Use species.name_suggest():
saurita_match = saurita_suggestions[0]

## Save the taxon key:
saurita_key = saurita_match["speciesKey"]

## Count georeferenced records:
saurita_count = occ.count(
    taxonKey=saurita_key,
    isGeoreferenced=True,
)

print(f"There are {saurita_count} occurrence records with lat/long information.")


# How many total T. saurita records are there?
saurita_nolatlong_count = occ.count(
    taxonKey=saurita_key,
    isGeoreferenced=False,
)

# print(f"There are {saurita_nolatlong_count} occurrence records without lat/long information.")

saurita_total_count = saurita_count + saurita_nolatlong_count

print(f"There are {saurita_total_count} total occurrence records available for \x1B[3mThamnophis saurita\x1B[0m.")

## Fetch up to 100 records using occ.search():
saurita_records = occ.search(
    speciesKey = saurita_key,
    hasCoordinate = True,
    hasGeospatialIssue = False,
    limit = 100,
)

saurita_df = pd.DataFrame(saurita_records["results"])

saurita_small = saurita_df[
    ["key", "scientificName", "decimalLongitude", "decimalLatitude", "year", "basisOfRecord", "countryCode"]
]

## Convert records to a GeoDataFrame:
saurita_gdf = gpd.GeoDataFrame(
    saurita_small,
    geometry = gpd.points_from_xy(saurita_small["decimalLongitude"], saurita_small["decimalLatitude"]),
    crs = "EPSG:4326",
)

saurita_gdf["Species"] = "Thamnophis saurita"

## Map the records:
saurita_gdf.explore()

There are 11195 occurrence records with lat/long information.
There are 14378 total occurrence records available for Thamnophis saurita.


In [64]:
## Map all four species together:
gbif_snakes_all = pd.concat([dekayi_gdf, contortrix_gdf, guttatus_gdf, saurita_gdf])
gbif_snakes_all.explore(column="Species", cmap="rainbow")

## 22. Written reflection

Answer these questions after running your code.

**Question 1:** Which of your species had the most GBIF records available?

**Your answer:** *Storeria dekayi*

- *Storeria dekayi* has 56546 total occurrence records available from GBIF.
- *Agkistrodon contortrix* has 29114 total occurrence records available from GBIF.
- *Pantherophis guttatus* has 13207 total occurrence records available from GBIF.
- *Thamnophis saurita* has 14378 total occurrence records available from GBIF.

**Question 2:** Did the GBIF points look similar to the local CSV points from Assignment 01? Why might GBIF records look different?

**Your answer:** The GBIF points show similar species distributions to the data in the EasternSnakes files, but the exact points are not the same (based only on visual inspection). In this current assignment, we've taken a subset (less than 1%) of the occurrence records available in GBIF so our representation of the species distribution may be biased through the data collection process. I want to know a few things:
- If we limit our download to 100 records, are the selected records randomly downloaded from all available records? Or, does it work down the list sequentially (potentially biased for sample time, upload date, observation lat/long, etc.)
- Can we confirm whether or not the EasternSnakes data has been uploaded to GBIF? If so, I would expect those exact points to be matched if we download the full record list (unless some sort of rounding is being applied to location data - might be used for rare species and/or private data in human-subject datasets). 

**Question 3:** What is one reason it is useful to count records before downloading or mapping them?

**Your answer:** You want to compare the total available records with what you have actually downloaded to make sure you are receiving the data that you expect. If there has been an error in the download process, or you've made an error in your filter selection, then you may not get the data that you are expecting. Counting the records and comparing to what you expect is a easy first check. 


## 23. Submit your work

Before submitting, make sure you have run the notebook from top to bottom and answered the written questions.

Commit and push your completed notebook to your class GitHub repository.

Open a terminal window and run these commands to add, commit, and push your notebook:

```
# Go to the labs directory in the course repo
cd ~/BIO597-SpatialBiodiversity/docs/assignments

# Add your changed lab
git add Assignment-02-pygbif.ipynb
git commit -m 'Finished Assignment 02'
git push
```